# LeetCode 2070 — Most Beautiful Item for Each Query bằng Python

Link đề: https://leetcode.com/problems/most-beautiful-item-for-each-query/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Sorting

Skeleton LeetCode:

```python
class Solution:
    def maximumBeauty(self, items: List[List[int]], queries: List[int]) -> List[int]:
        ...
```

## 1. Mục tiêu học

1. Biến truy vấn "max beauty với `price ≤ q`" thành **max tiền tố** trên danh sách đã sắp theo giá, rồi tìm vị trí bằng **binary search** (`bisect_right`).
2. Cách **ngoại tuyến** (offline): sắp cả truy vấn, quét hai con trỏ, ghi kết quả về **đúng thứ tự gốc**.
3. Xử lý giá trùng nhau (`bisect_right` để lấy **mọi** món có `price ≤ q`) và trường hợp không có món nào (`0`).
4. Phân tích `O((n + q) log n)` so với vét cạn `O(n · q)`.

## 2. Đề bài & ví dụ

Cho mảng `items`, `items[i] = [price_i, beauty_i]`, và mảng `queries`. Với mỗi `queries[j]`, tìm **beauty lớn nhất** của một món có `price ≤ queries[j]` (0 nếu không có). Trả về mảng đáp án cùng độ dài, **theo đúng thứ tự** `queries`.

**Ví dụ 1:** `items = [[1,2],[3,2],[2,4],[5,6],[3,5]]`, `queries = [1,2,3,4,5,6]` → `[2,4,5,5,6,6]`.

**Ví dụ 2:** `items = [[1,2],[1,2],[1,3],[1,4]]`, `queries = [1]` → `[4]`.

**Ví dụ 3:** `items = [[10,1000]]`, `queries = [5]` → `[0]`.

**Ràng buộc:** `1 ≤ n, q ≤ 10⁵`; `1 ≤ price, beauty, query ≤ 10⁹`.

In [1]:
import random
import time
import bisect
from typing import List

examples = [(([[1, 2], [3, 2], [2, 4], [5, 6], [3, 5]], [1, 2, 3, 4, 5, 6]), [2, 4, 5, 5, 6, 6]),
            (([[1, 2], [1, 2], [1, 3], [1, 4]], [1]), [4]),
            (([[10, 1000]], [5]), [0])]
for (it, qs), exp in examples:
    print(f"items={it}, queries={qs} -> kỳ vọng {exp}")

items=[[1, 2], [3, 2], [2, 4], [5, 6], [3, 5]], queries=[1, 2, 3, 4, 5, 6] -> kỳ vọng [2, 4, 5, 5, 6, 6]
items=[[1, 2], [1, 2], [1, 3], [1, 4]], queries=[1] -> kỳ vọng [4]
items=[[10, 1000]], queries=[5] -> kỳ vọng [0]


## 3. Kiến thức nền

### 3.1 Max tiền tố

`best[i] = max(b[0], …, b[i])` không giảm theo `i`. Nếu sắp món theo giá tăng dần, tập món có `price ≤ q` là một **tiền tố** ⇒ đáp án là `best` tại cuối tiền tố đó.

In [2]:
items = sorted(examples[0][0][0])
prices = [p for p, _ in items]
best, cur = [], 0
for _, b in items:
    cur = max(cur, b); best.append(cur)
print("sắp theo giá:", items); print("prices:", prices, "| best:", best)

sắp theo giá: [[1, 2], [2, 4], [3, 2], [3, 5], [5, 6]]
prices: [1, 2, 3, 3, 5] | best: [2, 4, 4, 5, 6]


### 3.2 `bisect_right` cho "số phần tử `≤ q`"

Trên `prices` đã sắp, `bisect_right(prices, q)` = số món có giá `≤ q` (kể cả trùng giá). Nếu bằng 0 ⇒ không có món nào.

In [3]:
for q in (0, 1, 3, 4, 10):
    k = bisect.bisect_right(prices, q)
    print(f"q={q}: {k} món giá <= q -> {best[k - 1] if k else 0}")

q=0: 0 món giá <= q -> 0
q=1: 1 món giá <= q -> 2
q=3: 4 món giá <= q -> 5
q=4: 4 món giá <= q -> 5
q=10: 5 món giá <= q -> 6


### 3.3 Sắp xếp kèm chỉ số gốc

Xử lý ngoại tuyến cần nhớ vị trí gốc: `sorted(range(q), key=lambda j: queries[j])`.

### 3.4 Giá trị lớn (`10⁹`)

Không thể lập mảng theo giá; làm việc trên các giá xuất hiện (sắp xếp + bisect) hoặc nén tọa độ.

## 4. Ý tưởng nền tảng / trực giác

Mỗi truy vấn hỏi max trên một **tiền tố** (theo giá). Sắp một lần `O(n log n)`, tính max tiền tố `O(n)`, rồi mỗi truy vấn tìm điểm cắt bằng binary search `O(log n)`.

Tính tay **Ví dụ 1** (sau khi sắp: `[1,2],[2,4],[3,2],[3,5],[5,6]`, `best = [2,4,4,5,6]`):

| q | bisect_right | best | đáp án |
|---|---|---|---|
| 1 | 1 | best[0] = 2 | 2 |
| 2 | 2 | best[1] = 4 | 4 |
| 3 | 4 | best[3] = 5 | 5 |
| 4 | 4 | 5 | 5 |
| 5 | 5 | 6 | 6 |
| 6 | 5 | 6 | 6 |

## 5. Cách 1 — Brute Force / baseline: duyệt mọi món cho mỗi truy vấn

In [4]:
def beauty_bruteforce(items, queries):
    return [max((b for p, b in items if p <= q), default=0) for q in queries]

for (it, qs), exp in examples:
    got = beauty_bruteforce(it, qs); print(got, "kỳ vọng", exp); assert got == exp

[2, 4, 5, 5, 6, 6] kỳ vọng [2, 4, 5, 5, 6, 6]
[4] kỳ vọng [4]
[0] kỳ vọng [0]


**Độ phức tạp:** `O(n · q)` = `10¹⁰` ⇒ TLE.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** tập món hợp lệ cho `q` là tiền tố sau khi sắp theo giá.

**Insight 2:** max tiền tố trả lời "max trên tiền tố" `O(1)`.

**Insight 3:** `bisect_right(prices, q)` cho độ dài tiền tố `O(log n)`.

**Insight 4 (ngoại tuyến):** sắp truy vấn tăng dần; con trỏ trên món chỉ tiến, `cur = max beauty đã gặp` ⇒ tổng `O(n log n + q log q)` không cần bisect.

**Invariant (ngoại tuyến):** khi xử lý truy vấn `q` (theo thứ tự tăng), `cur` là max beauty của mọi món có giá `≤ q`.

### 6.1 Minh họa: vì sao cần max **tiền tố**, không chỉ món cuối

Món đắt hơn chưa chắc đẹp hơn: ở Ví dụ 1, món `[3,2]` đứng sau `[2,4]` nhưng beauty nhỏ hơn. Nếu lấy beauty của món **cuối** trong tiền tố, kết quả phụ thuộc thứ tự sắp các món cùng giá (với `key=lambda x: x[0]` và thứ tự gốc, truy vấn `q = 3` ra `5`; nếu đảo hai món giá 3 thì ra `2`), và hoàn toàn sai khi món rẻ hơn lại đẹp hơn.

In [5]:
for q in (2, 3):
    k = bisect.bisect_right(prices, q)
    print(f"q={q}: món cuối {items[k - 1]} -> {items[k - 1][1]} | max tiền tố -> {best[k - 1]}")

q=2: món cuối [2, 4] -> 4 | max tiền tố -> 4
q=3: món cuối [3, 5] -> 5 | max tiền tố -> 5


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp + max tiền tố + `bisect` (bản nộp)

In [6]:
class Solution:
    def maximumBeauty(self, items: List[List[int]], queries: List[int]) -> List[int]:
        items = sorted(items)
        prices = [p for p, _ in items]
        best = []
        cur = 0
        for _, b in items:
            cur = max(cur, b)
            best.append(cur)
        res = []
        for q in queries:
            k = bisect.bisect_right(prices, q)
            res.append(best[k - 1] if k else 0)
        return res

print([Solution().maximumBeauty(*a) for a, _ in examples])

[[2, 4, 5, 5, 6, 6], [4], [0]]


**Độ phức tạp Cách 2:** `O(n log n + q log n)` thời gian, `O(n)` bộ nhớ.

### 7.2 Cách 3 — Ngoại tuyến: sắp truy vấn + hai con trỏ

In [7]:
def beauty_offline(items, queries):
    items = sorted(items); res = [0] * len(queries)
    i = cur = 0
    for j in sorted(range(len(queries)), key=queries.__getitem__):
        q = queries[j]
        while i < len(items) and items[i][0] <= q:
            cur = max(cur, items[i][1]); i += 1
        res[j] = cur
    return res

print([beauty_offline(*a) for a, _ in examples])

[[2, 4, 5, 5, 6, 6], [4], [0]]


**Độ phức tạp Cách 3:** `O(n log n + q log q)`.

### 7.3 Cách 4 — Nén giá trùng: giữ "bậc thang" giá tăng & beauty tăng nghiêm ngặt

Sau khi sắp, chỉ giữ các món mà beauty **lớn hơn hẳn** mọi món rẻ hơn — danh sách "Pareto" có giá tăng và beauty tăng. Truy vấn = beauty của bậc cuối có giá `≤ q`. Danh sách có thể ngắn hơn nhiều so với `n`.

In [8]:
def beauty_staircase(items, queries):
    stair_p, stair_b = [], []
    for p, b in sorted(items):
        if not stair_b or b > stair_b[-1]:
            if stair_p and stair_p[-1] == p:
                stair_b[-1] = b                      # cùng giá: giữ beauty lớn hơn
            else:
                stair_p.append(p); stair_b.append(b)
    out = []
    for q in queries:
        k = bisect.bisect_right(stair_p, q)
        out.append(stair_b[k - 1] if k else 0)
    return out

print([beauty_staircase(*a) for a, _ in examples])

[[2, 4, 5, 5, 6, 6], [4], [0]]


## 8. TRACE từng bước

Trace Cách 3 (ngoại tuyến) trên **Ví dụ 1** với truy vấn xáo trộn `[5, 1, 3, 6, 2, 4]` để thấy kết quả được ghi về đúng vị trí gốc:

In [9]:
def trace_offline(items, queries):
    items = sorted(items); res = [0] * len(queries); i = cur = 0
    for j in sorted(range(len(queries)), key=queries.__getitem__):
        q = queries[j]; taken = []
        while i < len(items) and items[i][0] <= q:
            cur = max(cur, items[i][1]); taken.append(items[i]); i += 1
        res[j] = cur
        print(f"truy vấn #{j} (q={q}): thêm {taken} -> cur = {cur} -> res[{j}] = {cur}")
    return res

r = trace_offline(examples[0][0][0], [5, 1, 3, 6, 2, 4]); print(r); assert r == [6, 2, 5, 6, 4, 5]

truy vấn #1 (q=1): thêm [[1, 2]] -> cur = 2 -> res[1] = 2
truy vấn #4 (q=2): thêm [[2, 4]] -> cur = 4 -> res[4] = 4
truy vấn #2 (q=3): thêm [[3, 2], [3, 5]] -> cur = 5 -> res[2] = 5
truy vấn #5 (q=4): thêm [] -> cur = 5 -> res[5] = 5
truy vấn #0 (q=5): thêm [[5, 6]] -> cur = 6 -> res[0] = 6
truy vấn #3 (q=6): thêm [] -> cur = 6 -> res[3] = 6
[6, 2, 5, 6, 4, 5]


| thứ tự xử lý | q | món thêm vào | cur | ghi vào |
|---|---|---|---|---|
| 1 | 1 | `[1,2]` | 2 | res[1] |
| 2 | 2 | `[2,4]` | 4 | res[4] |
| 3 | 3 | `[3,2]`, `[3,5]` | 5 | res[2] |
| 4 | 4 | — | 5 | res[5] |
| 5 | 5 | `[5,6]` | 6 | res[0] |
| 6 | 6 | — | 6 | res[3] |

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2**.

In [10]:
class Solution:
    def maximumBeauty(self, items: List[List[int]], queries: List[int]) -> List[int]:
        items = sorted(items)
        prices = [p for p, _ in items]
        best = []
        cur = 0
        for _, b in items:
            cur = max(cur, b)
            best.append(cur)
        res = []
        for q in queries:
            k = bisect.bisect_right(prices, q)
            res.append(best[k - 1] if k else 0)
        return res

for (it, qs), exp in examples:
    got = Solution().maximumBeauty(it, qs); print(got, exp); assert got == exp

[2, 4, 5, 5, 6, 6] [2, 4, 5, 5, 6, 6]
[4] [4]
[0] [0]


## 10. Vì sao thuật toán đúng

Sắp `items` theo giá tăng (giá trùng theo thứ tự tùy ý). Với truy vấn `q`, đặt `k = bisect_right(prices, q)`: các món `0..k−1` có giá `≤ q`, các món `k..n−1` có giá `> q` (vì `prices` không giảm). Vậy tập món hợp lệ đúng là tiền tố độ dài `k`, và max beauty của nó là `best[k−1]` (định nghĩa max tiền tố); `k = 0` ⇒ không có món ⇒ `0`. ∎

**Ngoại tuyến:** truy vấn xử lý theo `q` tăng; con trỏ `i` dừng ở món đầu tiên có giá `> q` ⇒ `cur` = max của mọi món giá `≤ q` (quy nạp vì tập này chỉ lớn dần). Ghi `res[j]` theo chỉ số gốc ⇒ đúng thứ tự.

### 10.1 Kiểm chứng thực nghiệm (nhiều giá trùng nhau)

In [11]:
rng = random.Random(1010)
for _ in range(2000):
    it = [[rng.randint(1, 8), rng.randint(1, 10)] for _ in range(rng.randint(1, 8))]
    qs = [rng.randint(0, 10) for _ in range(rng.randint(1, 8))]
    ref = beauty_bruteforce(it, qs)
    assert ref == Solution().maximumBeauty(it, qs) == beauty_offline(it, qs) == beauty_staircase(it, qs)
print("4 cách khớp nhau trên 2000 bộ (giá trùng, truy vấn không có món)")

4 cách khớp nhau trên 2000 bộ (giá trùng, truy vấn không có món)


## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n·q)` | `O(1)` |
| 2. Max tiền tố + bisect | `O((n + q) log n)` | `O(n)` |
| 3. Ngoại tuyến | `O(n log n + q log q)` | `O(n + q)` |
| 4. Bậc thang Pareto | `O(n log n + q log s)` | `O(s)` |

In [12]:
rng = random.Random(2070)
it = [[rng.randint(1, 10 ** 9), rng.randint(1, 10 ** 9)] for _ in range(10 ** 5)]
qs = [rng.randint(1, 10 ** 9) for _ in range(10 ** 5)]
for name, f in [("bisect", Solution().maximumBeauty), ("offline", beauty_offline), ("staircase", beauty_staircase)]:
    t = time.perf_counter(); r = f(it, qs); print(f"{name:<9}: {(time.perf_counter() - t) * 1000:.0f} ms, tổng {sum(r)}")
t = time.perf_counter(); beauty_bruteforce(it[:2000], qs[:2000]); print(f"brute 2000 x 2000: {(time.perf_counter() - t) * 1000:.0f} ms")

bisect   : 196 ms, tổng 99983298753916
offline  : 194 ms, tổng 99983298753916


staircase: 92 ms, tổng 99983298753916
brute 2000 x 2000: 105 ms


## 12. Bộ test

In [13]:
ALL = {"bisect": Solution().maximumBeauty, "offline": beauty_offline, "staircase": beauty_staircase}
def run_all(it, qs, brute=True):
    res = {k: f([x[:] for x in it], list(qs)) for k, f in ALL.items()}
    if brute:
        res["brute"] = beauty_bruteforce(it, qs)
    vals = list(res.values()); assert all(v == vals[0] for v in vals), (it, qs, res)
    return vals[0]

for (it, qs), exp in examples:
    assert run_all(it, qs) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — kiểm bằng vét cạn trong run_all
edge = [(([[5, 1]], [5, 4, 6]), [1, 0, 1]), (([[1, 5], [2, 1]], [2]), [5]), (([[2, 3], [2, 7], [2, 1]], [1, 2]), [0, 7]),
        (([[10 ** 9, 10 ** 9]], [10 ** 9]), [10 ** 9]), (([[3, 1], [1, 9]], [3, 2, 1]), [9, 9, 9])]
for (it, qs), exp in edge:
    assert run_all(it, qs) == exp, (it, qs, run_all(it, qs))
rng = random.Random(20700)
for _ in range(400):
    run_all([[rng.randint(1, 50), rng.randint(1, 50)] for _ in range(rng.randint(1, 20))], [rng.randint(1, 60) for _ in range(rng.randint(1, 20))])
print("Ví dụ, 5 edge case, 400 bộ ngẫu nhiên: OK")

Ví dụ, 5 edge case, 400 bộ ngẫu nhiên: OK


### 12.1 Kiểm tra tính chất

- Đáp án **không giảm** theo `q`.
- Hoán vị `items` không đổi đáp án; hoán vị `queries` hoán vị đáp án tương ứng.
- Với `q ≥ max price`, đáp án là `max beauty`.

In [14]:
rng = random.Random(121)
f = Solution().maximumBeauty
for _ in range(500):
    it = [[rng.randint(1, 30), rng.randint(1, 30)] for _ in range(rng.randint(1, 12))]
    qs = sorted(rng.randint(1, 35) for _ in range(10)); a = f(it, qs)
    assert all(a[i] <= a[i + 1] for i in range(len(a) - 1))
    sh = [x[:] for x in it]; rng.shuffle(sh); assert f(sh, qs) == a
    perm = list(range(10)); rng.shuffle(perm); assert f(it, [qs[p] for p in perm]) == [a[p] for p in perm]
    assert f(it, [max(p for p, _ in it)]) == [max(b for _, b in it)]
print("4 tính chất đúng trên 500 bộ")

4 tính chất đúng trên 500 bộ


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng beauty của món **cuối** tiền tố (thiếu max tiền tố)

In [15]:
def wrong_last_item(items, queries):
    items = sorted(items); prices = [p for p, _ in items]
    return [items[k - 1][1] if (k := bisect.bisect_right(prices, q)) else 0 for q in queries]   # SAI

for (it, qs), exp in examples:
    print("sai :", wrong_last_item(it, qs), "| đúng:", exp, "(trùng vì sorted() xếp cùng giá theo beauty tăng)")
it, qs = [[1, 9], [2, 1]], [2]        # tự dựng (đối chiếu lại đề gốc): món rẻ hơn lại đẹp hơn
print("sai :", wrong_last_item(it, qs), "| đúng:", Solution().maximumBeauty(it, qs), "| brute:", beauty_bruteforce(it, qs))

sai : [2, 4, 5, 5, 6, 6] | đúng: [2, 4, 5, 5, 6, 6] (trùng vì sorted() xếp cùng giá theo beauty tăng)
sai : [4] | đúng: [4] (trùng vì sorted() xếp cùng giá theo beauty tăng)
sai : [0] | đúng: [0] (trùng vì sorted() xếp cùng giá theo beauty tăng)
sai : [1] | đúng: [9] | brute: [9]


**Vì sao sai:** món đắt nhất trong tầm giá không nhất thiết đẹp nhất. Ba ví dụ của đề tình cờ đúng vì `sorted()` xếp các món cùng giá theo beauty tăng, nhưng với `[[1,9],[2,1]]`, truy vấn `2` cho `1` thay vì `9`. **Sửa:** max tiền tố.

### Lỗi 2 — `bisect_left` (bỏ món có giá **đúng bằng** `q`)

In [16]:
def wrong_bisect_left(items, queries):
    items = sorted(items); prices = [p for p, _ in items]; best = []; cur = 0
    for _, b in items: cur = max(cur, b); best.append(cur)
    return [best[k - 1] if (k := bisect.bisect_left(prices, q)) else 0 for q in queries]   # SAI

for (it, qs), exp in examples:
    print("sai :", wrong_bisect_left(it, qs), "| đúng:", exp)

sai : [0, 2, 4, 5, 5, 6] | đúng: [2, 4, 5, 5, 6, 6]
sai : [0] | đúng: [4]
sai : [0] | đúng: [0]


**Vì sao sai:** `bisect_left` đếm số món giá **`< q`**; đề cho phép `≤ q`. **Sửa:** `bisect_right`.

### Lỗi 3 — Ngoại tuyến nhưng trả kết quả theo thứ tự **đã sắp**

In [17]:
def wrong_order(items, queries):
    items = sorted(items); out = []; i = cur = 0
    for q in sorted(queries):                                   # SAI: mất thứ tự gốc
        while i < len(items) and items[i][0] <= q: cur = max(cur, items[i][1]); i += 1
        out.append(cur)
    return out

it, qs = examples[0][0][0], [5, 1, 3]        # truy vấn xáo trộn (tự dựng, đối chiếu lại đề gốc)
print("sai :", wrong_order(it, qs), "| đúng:", Solution().maximumBeauty(it, qs))

sai : [2, 5, 6] | đúng: [6, 2, 5]


**Vì sao sai:** đáp án phải cùng thứ tự với `queries`. **Sửa:** sắp **chỉ số** và ghi `res[j]`.

### Lỗi 4 — Trả `-1` (hoặc lỗi chỉ số) khi không có món nào

In [18]:
def wrong_no_item(items, queries):
    items = sorted(items); prices = [p for p, _ in items]; best = []; cur = 0
    for _, b in items: cur = max(cur, b); best.append(cur)
    return [best[bisect.bisect_right(prices, q) - 1] for q in queries]          # SAI: k = 0 -> best[-1]

for (it, qs), exp in examples:
    print("sai :", wrong_no_item(it, qs), "| đúng:", exp)

sai : [2, 4, 5, 5, 6, 6] | đúng: [2, 4, 5, 5, 6, 6]
sai : [4] | đúng: [4]
sai : [1000] | đúng: [0]


**Vì sao sai:** khi `k = 0`, `best[-1]` là phần tử **cuối** (max toàn bộ) — Python không báo lỗi. Ví dụ 3 bản sai trả `1000`. **Sửa:** `best[k − 1] if k else 0`.

### Lỗi 5 — Sắp theo beauty hoặc không sắp

In [19]:
def wrong_unsorted(items, queries):
    prices = [p for p, _ in items]; best = []; cur = 0
    for _, b in items: cur = max(cur, b); best.append(cur)                      # SAI: items chưa sắp theo giá
    return [best[k - 1] if (k := bisect.bisect_right(prices, q)) else 0 for q in queries]

for (it, qs), exp in examples:
    print("sai :", wrong_unsorted(it, qs), "| đúng:", exp)

sai : [2, 4, 6, 6, 6, 6] | đúng: [2, 4, 5, 5, 6, 6]
sai : [4] | đúng: [4]
sai : [0] | đúng: [0]


**Vì sao sai:** `bisect` chỉ đúng trên mảng **đã sắp**; tiền tố theo chỉ số gốc không phải tập "giá `≤ q`". **Sửa:** `sorted(items)` (theo giá).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Với mỗi truy vấn, max/min/tổng trên các phần tử có khóa `≤ q`" ⇒ **sắp theo khóa + tiền tố + bisect**, hoặc **ngoại tuyến** sắp truy vấn.
- `bisect_right` cho `≤`, `bisect_left` cho `<`.
- Ngoại tuyến: luôn ghi kết quả theo **chỉ số gốc**.
- Python: chỉ số `-1` không lỗi — kiểm tra trường hợp rỗng.

## 15. Biến thể của bài

1. **Truy vấn khoảng giá `[lo, hi]`** ⇒ cần max trên đoạn (sparse table / segment tree) sau khi sắp (Bài tập 5).
2. **Cập nhật động** (thêm món) ⇒ Fenwick max trên giá đã nén (chỉ tăng).
3. **1847 Closest Room**: ngoại tuyến + `SortedList`.
4. **2512 / 2251 Number of Flowers in Full Bloom**: sắp + bisect.
5. **Đếm số món** có `price ≤ q` và `beauty ≥ b` ⇒ ngoại tuyến 2 chiều + Fenwick.

### 15.1 Code biến thể: thêm món động với Fenwick max (giá nén ngoại tuyến)

In [20]:
class FenwickMax:
    def __init__(self, n): self.n = n; self.t = [0] * (n + 1)
    def update(self, i, v):
        i += 1
        while i <= self.n: self.t[i] = max(self.t[i], v); i += i & -i
    def query(self, i):                 # max trên [0, i)
        r = 0
        while i > 0: r = max(r, self.t[i]); i -= i & -i
        return r

def dynamic_beauty(ops):
    # ops: ("add", price, beauty) hoặc ("ask", q); giá nén ngoại tuyến
    coords = sorted({op[1] for op in ops if op[0] == "add"})
    fw = FenwickMax(len(coords)); out = []
    for op in ops:
        if op[0] == "add": fw.update(bisect.bisect_left(coords, op[1]), op[2])
        else: out.append(fw.query(bisect.bisect_right(coords, op[1])))
    return out

rng = random.Random(15)
for _ in range(300):
    ops, items_now, exp = [], [], []
    for _ in range(rng.randint(1, 20)):
        if rng.random() < 0.5:
            p, b = rng.randint(1, 20), rng.randint(1, 30); ops.append(("add", p, b)); items_now.append((p, b))
        else:
            q = rng.randint(0, 22); ops.append(("ask", q)); exp.append(max((b for p, b in items_now if p <= q), default=0))
    assert dynamic_beauty(ops) == exp
print("Phiên bản động (Fenwick max) khớp vét cạn trên 300 chuỗi thao tác")

Phiên bản động (Fenwick max) khớp vét cạn trên 300 chuỗi thao tác


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Duyệt mọi món | `O(nq)` | `O(1)` | Kiểm chứng |
| 2. Tiền tố + bisect | Trực tuyến | `O((n+q) log n)` | `O(n)` | **Bản nộp** |
| 3. Ngoại tuyến | Hai con trỏ | `O(n log n + q log q)` | `O(n+q)` | Không cần bisect |
| 4. Bậc thang | Lọc Pareto | `O(n log n + q log s)` | `O(s)` | Nhiều món bị "trội" |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `items = [[3,1],[1,9]]`, `queries = [3, 2, 1, 0]`.

**Bài 2 (tính tay):** Với `items = [[2,3],[2,7],[2,1],[4,5],[6,8]]`, lập `prices`, `best` và trả lời `q = 1, 2, 5, 6`.

**Bài 3 (điền TODO):** Hoàn thiện `prefix_max(values)`.

**Bài 4 (tự viết lại):** Viết `min_price_for_beauty(items, targets)`: với mỗi `t`, giá **nhỏ nhất** của một món có `beauty ≥ t` (−1 nếu không có).

**Bài 5 (mini challenge):** Truy vấn `[lo, hi]`: max beauty của món có `lo ≤ price ≤ hi`. Dùng sparse table trên beauty đã sắp theo giá; `O((n + q) log n)`; đối chiếu vét cạn.

In [21]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def prefix_max(values):
    out, cur = [], 0
    # TODO: với mỗi v: cur = max(cur, v); out.append(cur)
    return out

In [22]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def min_price_for_beauty(items, targets):
    # TODO: sắp theo beauty giảm dần, min hậu tố theo giá, bisect theo beauty
    pass

def range_beauty(items, queries):
    # TODO: sắp theo giá; sparse table max; i = bisect_left(prices, lo), j = bisect_right(prices, hi) - 1
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: sắp: `[1,9],[3,1]`, `best = [9, 9]`. Mức 2: `q = 3, 2, 1` đều có món giá 1. Mức 3: `[9, 9, 9, 0]`.

**Bài 2.** Mức 1: `prices = [2,2,2,4,6]`. Mức 2: sau khi sắp `[2,1],[2,3],[2,7],[4,5],[6,8]`: `best = [1,3,7,7,8]`; cuối nhóm giá 2 là 7. Mức 3: `q=1 → 0`, `q=2 → 7`, `q=5 → 7`, `q=6 → 8`.

**Bài 3.** Mức 1: biến chạy `cur`. Mức 2: cập nhật max. Mức 3: thêm vào danh sách.

**Bài 4.** Mức 1: món có beauty `≥ t` là một **hậu tố** khi sắp theo beauty tăng. Mức 2: min giá hậu tố. Mức 3: `bisect_left(beauties, t)`.

**Bài 5.** Mức 1: sắp theo giá, mảng beauty theo thứ tự đó. Mức 2: sparse table `O(n log n)`. Mức 3: truy vấn `O(1)` sau hai `bisect`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2, 3
f = Solution().maximumBeauty
b1 = f([[3, 1], [1, 9]], [3, 2, 1, 0]); print("Bài 1:", b1); assert b1 == [9, 9, 9, 0]
b2 = f([[2, 3], [2, 7], [2, 1], [4, 5], [6, 8]], [1, 2, 5, 6]); print("Bài 2:", b2); assert b2 == [0, 7, 7, 8]
def prefix_max(values):
    out, cur = [], 0
    for v in values:
        cur = max(cur, v); out.append(cur)
    return out
assert prefix_max([2, 4, 2, 5, 6]) == [2, 4, 4, 5, 6]
print("Bài 3: OK")

Bài 1: [9, 9, 9, 0]
Bài 2: [0, 7, 7, 8]
Bài 3: OK


In [24]:
# Bài 4, 5
def min_price_for_beauty(items, targets):
    it = sorted(items, key=lambda x: x[1]); beauties = [b for _, b in it]
    suf = [float("inf")] * (len(it) + 1)
    for i in range(len(it) - 1, -1, -1): suf[i] = min(suf[i + 1], it[i][0])
    out = []
    for t in targets:
        k = bisect.bisect_left(beauties, t); out.append(-1 if suf[k] == float("inf") else suf[k])
    return out

def range_beauty(items, queries):
    it = sorted(items); prices = [p for p, _ in it]; arr = [b for _, b in it]; n = len(arr)
    sp = [arr[:]]; k = 1
    while (1 << k) <= n:
        prev = sp[-1]; h = 1 << (k - 1); sp.append([max(prev[i], prev[i + h]) for i in range(n - (1 << k) + 1)]); k += 1
    out = []
    for lo, hi in queries:
        i = bisect.bisect_left(prices, lo); j = bisect.bisect_right(prices, hi) - 1
        if i > j: out.append(0); continue
        lv = (j - i + 1).bit_length() - 1; out.append(max(sp[lv][i], sp[lv][j - (1 << lv) + 1]))
    return out

rng = random.Random(5)
for _ in range(300):
    it = [[rng.randint(1, 20), rng.randint(1, 20)] for _ in range(rng.randint(1, 10))]
    ts = [rng.randint(1, 22) for _ in range(5)]
    assert min_price_for_beauty(it, ts) == [min((p for p, b in it if b >= t), default=-1) for t in ts]
    qs = [sorted((rng.randint(0, 22), rng.randint(0, 22))) for _ in range(5)]
    assert range_beauty(it, qs) == [max((b for p, b in it if lo <= p <= hi), default=0) for lo, hi in qs]
print("Bài 4, 5: OK |", min_price_for_beauty(examples[0][0][0], [5, 6, 7]), range_beauty(examples[0][0][0], [[2, 3], [4, 4]]))

Bài 4, 5: OK | [3, 5, -1] [5, 0]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
items.sort()                                     # theo giá
prices = [p for p, _ in items]
best = max tiền tố của beauty
for q in queries:
    k = bisect_right(prices, q)                  # số món giá <= q
    ans.append(best[k - 1] if k else 0)
```

**Checklist:**
- [ ] Sắp theo **giá** trước khi bisect?
- [ ] Max **tiền tố** (không phải beauty của món cuối)?
- [ ] `bisect_right` cho `≤ q`?
- [ ] `k = 0` ⇒ `0` (không dùng `best[-1]`)?
- [ ] Ngoại tuyến: ghi theo **chỉ số gốc**?